In [0]:
from pyspark.sql import functions as F

catalog, schema = "workspace", "patient_notes"
silver_table = f"{catalog}.{schema}.silver_notes"
gold_table   = f"{catalog}.{schema}.gold_note_conditions"
alias_table  = f"{catalog}.{schema}.ref_condition_aliases"

endpoint = "databricks-meta-llama-3-3-70b-instruct"   # <- replace with the name from your Serving page

display(spark.sql(f"SELECT ai_query('{endpoint}', 'Reply with exactly three words: pipeline is ready') AS reply"))

In [0]:
names = sorted(r.standard_name for r in spark.table(alias_table).select("standard_name").distinct().collect())

instructions = f"""You read the assessment section of a clinical note.
List every medical condition mentioned, with its status.
Use ONLY these standard condition names: {", ".join(names)}.
Map abbreviations and synonyms to the standard name (for example HTN -> hypertension).
Status must be exactly one of:
- current: active now; found or treated at this visit
- pre_existing: diagnosed before and still present (for example "known case of")
- history: in the past, may be resolved
- operated: treated surgically in the past (for example "s/p", "status post", "operated for")
- ruled_out: the doctor says the patient does not have it
Assessment: """

print(instructions)
print(len(instructions)) 

In [0]:
silver = spark.table(silver_table)

if spark.catalog.tableExists(gold_table):
    to_do = silver.join(spark.table(gold_table).select("note_id"), "note_id", "left_anti")   # skip already-done notes
else:
    to_do = silver

batch = to_do.orderBy("note_id").limit(20)
print("Notes in this batch:", batch.count())

In [0]:
ai_schema = "conditions ARRAY<STRUCT<condition:STRING, status:STRING>>"
# ai_query requires every top-level field of responseFormat to be a STRUCT, so wrap the array
response_format = "STRUCT<conditions_extraction:STRUCT<conditions:ARRAY<STRUCT<condition:STRING, status:STRING>>>>"

result = (batch
    .withColumn("request", F.concat(F.lit(instructions), F.col("assessment")))
    .withColumn("ai", F.expr(
        f"ai_query('{endpoint}', request, responseFormat => '{response_format}', failOnError => false)"))
    .withColumn("parsed", F.from_json(F.col("ai.result"), ai_schema))
    .select("note_id", "patient_id", "note_date", "assessment",
            F.col("parsed.conditions").alias("conditions"),
            F.col("ai.errorMessage").alias("ai_error"),
            F.current_timestamp().alias("extracted_at")))

result.write.mode("append").saveAsTable(gold_table)

gold = spark.table(gold_table)
print("Gold rows:", gold.count(), "| errors:", gold.filter("ai_error IS NOT NULL").count())
display(gold.select("note_id", "assessment", "conditions", "ai_error"))